In [3]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os
import requests
PASSWORD = os.getenv('SNOWSQL_PWD')
print(PASSWORD)

5eWxWv4EvyCkhkY


In [141]:
try:
    ctx = snowflake.connector.connect(
        user='ABHINAVSHARMA2002',
        password=PASSWORD,
        account='qraojwa-yg67137'
    )
    cs = ctx.cursor()
    try:
        cs.execute("CREATE WAREHOUSE IF NOT EXISTS task_1_warehouse_mg")
        cs.execute("CREATE DATABASE IF NOT EXISTS testdb_mg")
        cs.execute("USE DATABASE testdb_mg")
        cs.execute("CREATE SCHEMA IF NOT EXISTS task_2_mg")
        cs.execute("USE WAREHOUSE task_1_warehouse_mg")
        cs.execute("USE SCHEMA task_2_mg")
        ##cs.execute("SHOW VIEWS IN task_2_mg")
        ##print(cs.fetchall())
        runQueries(cs)
        cs.execute('SELECT * FROM "v_order_f" LIMIT 10')
        print(cs.fetchall())
    except Exception as e:
        print(f"Error: {e}") 
    finally:
        cs.close()
        ctx.close()
except Exception as e:
    print(f"Error: {e}")

[(406935, 1726790400000000000, 3, 'Tablet', 'Electronics', 1, 349.22, 349.22, 521.3854600000001, 2024, 3, 9, 38, 70, 'Gabriel Hebert', 'travisfrancis@yahoo.com', 'North Jesus', 'Germany'), (702411, 1726876800000000000, 19, 'Portable Projector', 'Electronics', 4, 885.96, 3543.84, 5290.953120000001, 2024, 3, 9, 38, 75, 'Kristy Schwartz', 'oaustin@hotmail.com', 'South Lauramouth', 'Australia'), (112236, 1727049600000000000, 15, 'Power Bank', 'Electronics', 2, 1736.19, 3472.38, 5184.26334, 2024, 3, 9, 39, 14, 'Kyle Anderson', 'brett58@yahoo.com', 'Pricebury', 'UK'), (906335, 1727136000000000000, 8, 'Mechanical Keyboard', 'Electronics', 4, 459.38, 1837.52, 2743.4173600000004, 2024, 3, 9, 39, 45, 'Hunter Schultz', 'katiedean@cunningham-keller.com', 'West Jeffreyfurt', 'France'), (654805, 1727222400000000000, 17, 'Wireless Router', 'Electronics', 2, 1206.46, 2412.92, 3602.4895600000004, 2024, 3, 9, 39, 24, 'Mitchell Ellis', 'newmanmichael@hotmail.com', 'South Matthewberg', 'Germany'), (948408

In [137]:
def runQueries(cs):    
    query = """
    CREATE OR REPLACE VIEW "v_order_f" AS
        SELECT 
            o."order_id",
            o."date",
            p."product_id",
            p."product_name",
            p."category",
            o."quantity",
            p."price",
            o."quantity" * p."price" AS "total_amount_local",
            (o."quantity" * p."price" * ex."exchange_rate") AS "total_amount_usd",
            
            -- Fiscal Attributes
            YEAR(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_year",
            QUARTER(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_quarter",
            MONTH(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_month",
            WEEK(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_week",

            -- Customer Information
            cust."customer_id",
            cust."customer_name",
            cust."email",
            cust."customer_city",
            cust."country"

        FROM orders o
        JOIN products p ON o."product_id" = p."product_id"
        JOIN customers cust ON o."customer_id" = cust."customer_id"
        JOIN exchangerates ex ON ex."target_currency" = 'USD'
    """

    cs.execute(query)